# v2 — Buoyancy and potential density (sigma0) on the particles, 1993-2013, all four seeds

Same monthly binned means as `v2.Monthly_means_z_temp_sal_v3.ipynb`, but for two properties computed
**on the particles** with `gsw`, using exactly the recipe of
`Analysis/particle_diagnostics/0.1.Diagnostics.ipynb`:

```
p    = gsw.p_from_z(-z, lat)        # z is positive-down depth
SA   = gsw.SA_from_SP(sal, p, lon, lat)
CT   = gsw.CT_from_pt(SA, temp)
rho  = gsw.rho(SA, CT, p)           # in-situ density
b    = -9.81 * (rho - 1025) / 1025  # in-situ buoyancy  [m s-2]
sigma0 = gsw.sigma0(SA, CT)         # potential density anomaly, ref 0 dbar [kg m-3]
```

Reference pressures are unchanged: in-situ `p` for the buoyancy, 0 dbar for sigma0, so the results are
directly comparable with the old run's `mean_b_month.zarr` / `mean_sigma0_month.zarr`.
The 1534 stores are streamed one by one; the bin grid is read from the file written by the z/temp/sal
notebook so the two sets of maps share a grid.

In [1]:
# --- papermill parameters ---------------------------------------------------
tracks_root  = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2"
seeds        = ["11098723", "45678", "67891", "78876"]   # all four seeds -> one 1993-2013 climatology
out_dir      = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis"
fig_dir      = "/work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/figures"
bins_file    = "v2.bins_v3.npz"   # bin edges shared by both v2 notebooks (written by the z/T/S one)
n_lon_bins   = 80        # number of lon bin EDGES (as in the template)
n_lat_bins   = 30        # number of lat bin EDGES
n_scan       = 120       # stores sampled to fix the bin edges (only used if bins_file is absent)
n_jobs       = 16        # parallel processes streaming the stores
max_stores   = 0         # 0 = all stores; >0 = test mode on the first N stores
drop_first_obs = True    # skip obs=0 (the release position), exactly as in the template
sal_min      = 0.0       # samples with sal <= sal_min are treated as invalid
temp_min     = 0.0       # samples with temp <= temp_min are treated as invalid


In [2]:
# Parameters
n_jobs = 32


In [3]:
import gsw
print('gsw', gsw.__version__)
import os, time, json
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from xhistogram.xarray import histogram as xhist
from joblib import Parallel, delayed
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cmocean as cm

os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
os.chdir(out_dir)
print("working in", os.getcwd())


gsw 3.6.19


working in /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis


## Stores

In [4]:
stores = []
for s in seeds:
    ss = sorted((Path(tracks_root) / f"tracks_v3_{s}").glob("Parcels_run_*.zarr"))
    print(f"seed {s}: {len(ss)} stores")
    stores += ss
if max_stores:
    stores = stores[:max_stores]
    print("TEST MODE: using only", len(stores), "stores")
print("total stores:", len(stores))


seed 11098723: 438 stores
seed 45678: 439 stores
seed 67891: 292 stores
seed 78876: 365 stores
total stores: 1534


## Bin edges (shared with the z/temp/sal notebook)

In [5]:
def _lonlat_range(f):
    d = xr.open_zarr(f)
    lo = d.lon.values; la = d.lat.values
    return (np.nanmin(lo), np.nanmax(lo), np.nanmin(la), np.nanmax(la))

if os.path.exists(bins_file):
    z = np.load(bins_file)
    lon_bins, lat_bins = z["lon_bins"], z["lat_bins"]
    print("loaded bin edges from", bins_file)
else:
    idx = np.linspace(0, len(stores) - 1, min(n_scan, len(stores))).astype(int)
    t0 = time.time()
    r = Parallel(n_jobs=n_jobs)(delayed(_lonlat_range)(stores[i]) for i in idx)
    r = np.array(r)
    lo0, lo1, la0, la1 = r[:, 0].min(), r[:, 1].max(), r[:, 2].min(), r[:, 3].max()
    # pad outward by ~2 bin widths so particles outside the scanned sample still fall inside
    dlo = (lo1 - lo0) / (n_lon_bins - 1); dla = (la1 - la0) / (n_lat_bins - 1)
    lon_bins = np.linspace(lo0 - 2 * dlo, lo1 + 2 * dlo, n_lon_bins)
    lat_bins = np.linspace(la0 - 2 * dla, la1 + 2 * dla, n_lat_bins)
    np.savez(bins_file, lon_bins=lon_bins, lat_bins=lat_bins)
    print(f"scanned {len(idx)} stores in {time.time()-t0:.0f}s -> saved {bins_file}")

lon_c = 0.5 * (lon_bins[1:] + lon_bins[:-1])
lat_c = 0.5 * (lat_bins[1:] + lat_bins[:-1])
NLON, NLAT = len(lon_c), len(lat_c)
print(f"lon bins: {lon_bins[0]:.2f} .. {lon_bins[-1]:.2f}  ({NLON} bins)")
print(f"lat bins: {lat_bins[0]:.2f} .. {lat_bins[-1]:.2f}  ({NLAT} bins)")


loaded bin edges from v2.bins_v3.npz
lon bins: -96.94 .. -14.67  (79 bins)
lat bins: -4.38 .. 32.02  (29 bins)


## Streaming: gsw on each store, then binned sum / count

In [6]:
import gsw
VARS = ["b", "sigma0"]

def _bin_one(f, lon_bins, lat_bins, vlist, drop_first_obs, sal_min, temp_min):
    """Stream ONE store: weighted xhistogram (binned sum) + binned count per variable.

    Exactly the template's estimator (Analysis/Monthly_means_z_temp_sal.ipynb):
        count = xhist(lon, lat, bins=[lon_bins, lat_bins], dim=["trajectory"])
        wsum  = xhist(lon, lat, bins=..., dim=["trajectory"], weights=var)
    with the coordinates masked where the variable is NaN, summed over obs.
    Counts and weighted sums are **additive**, so accumulating them store by store and
    dividing sum/count at the very end gives exactly the same monthly mean per bin as
    running the concatenated dataset in one go - but memory stays flat.
    Returns (month, sums, counts, stats) with tiny (NLON, NLAT) arrays.
    """
    ds = xr.open_zarr(f)
    month = int(pd.Timestamp(ds.time.values[0, 0]).month)
    sl = slice(1, None) if drop_first_obs else slice(None)   # skip obs=0 (release position)
    d = ds[["lon", "lat", "z", "temp", "sal"]].isel(obs=sl).compute()

    bad = (~np.isfinite(d.temp)) | (~np.isfinite(d.sal)) | (d.sal <= sal_min) | (d.temp <= temp_min)
    d["temp"] = d.temp.where(~bad)
    d["sal"]  = d.sal.where(~bad)

    # --- gsw recipe, identical to Analysis/particle_diagnostics/0.1.Diagnostics.ipynb ---
    p    = gsw.p_from_z(-d.z, d.lat)            # z is positive-down depth
    SA   = gsw.SA_from_SP(d.sal, p, d.lon, d.lat)
    CT   = gsw.CT_from_pt(SA, d.temp)
    rho  = gsw.rho(SA, CT, p)                   # in-situ density
    buoy = (-9.81 * (rho - 1025.0) / 1025.0).rename("b")   # in-situ buoyancy [m s-2]
    sig0 = gsw.sigma0(SA, CT).rename("sigma0")  # potential density anomaly, ref 0 dbar
    fields = {"b": buoy, "sigma0": sig0}

    sums, counts = {}, {}
    stats = {"n_bad": int(bad.values.sum()),
             "n_finite_pos": int((np.isfinite(d.lon.values) & np.isfinite(d.lat.values)).sum()),
             "lon_min": float(d.lon.min()), "lon_max": float(d.lon.max()),
             "lat_min": float(d.lat.min()), "lat_max": float(d.lat.max()),
             "n_out": 0}
    for v in vlist:
        w   = fields[v]
        lon = d.lon.where(w.notnull())      # drop samples with no value for this var
        lat = d.lat.where(w.notnull())
        count = xhist(lon, lat, bins=[lon_bins, lat_bins], dim=["trajectory"], bin_dim_suffix="")
        wsum  = xhist(lon, lat, bins=[lon_bins, lat_bins], dim=["trajectory"], weights=w,
                      bin_dim_suffix="")
        counts[v] = count.sum("obs").transpose("lon", "lat").values
        sums[v]   = wsum.sum("obs").transpose("lon", "lat").values

        vals = w.values.ravel()
        vals = vals[np.isfinite(vals) & np.isfinite(lon.values.ravel())
                    & np.isfinite(lat.values.ravel())]
        stats["n_" + v]   = int(vals.size)
        stats["n_out"]    = max(stats["n_out"], int(vals.size - counts[v].sum()))
        stats["min_" + v] = float(vals.min()) if vals.size else np.nan
        stats["max_" + v] = float(vals.max()) if vals.size else np.nan
        stats["sum_" + v] = float(vals.sum())
    return month, sums, counts, stats


In [7]:
SUM   = {v: np.zeros((12, NLON, NLAT)) for v in VARS}
COUNT = {v: np.zeros((12, NLON, NLAT)) for v in VARS}
all_stats = []

CHUNK = max(n_jobs * 4, 32)
t0 = time.time()
for i0 in range(0, len(stores), CHUNK):
    batch = stores[i0:i0 + CHUNK]
    res = Parallel(n_jobs=n_jobs)(
        delayed(_bin_one)(f, lon_bins, lat_bins, VARS, drop_first_obs, sal_min, temp_min)
        for f in batch)
    for month, sums, counts, st in res:
        for v in VARS:
            SUM[v][month - 1]   += sums[v]
            COUNT[v][month - 1] += counts[v]
        st["month"] = month
        all_stats.append(st)
    el = time.time() - t0
    done = i0 + len(batch)
    print(f"{done}/{len(stores)} stores  {el/60:.1f} min elapsed, "
          f"ETA {(el/done*(len(stores)-done))/60:.1f} min", flush=True)
print(f"streaming finished in {(time.time()-t0)/60:.1f} min")
stats = pd.DataFrame(all_stats)


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/coding/times.py:650: RuntimeWarning: invalid value encountered in cast
  flat_num = flat_num.astype(np.int64)


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


128/1534 stores  0.6 min elapsed, ETA 6.5 min


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/coding/times.py:650: RuntimeWarning: invalid value encountered in cast
  flat_num = flat_num.astype(np.int64)


256/1534 stores  1.2 min elapsed, ETA 5.8 min


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/coding/times.py:650: RuntimeWarning: invalid value encountered in cast
  flat_num = flat_num.astype(np.int64)


... (8 outputs left out)

1280/1534 stores  5.5 min elapsed, ETA 1.1 min


1408/1534 stores  6.1 min elapsed, ETA 0.5 min


1534/1534 stores  6.7 min elapsed, ETA 0.0 min


streaming finished in 6.7 min


## Sanity checks

`sigma0` is the potential density **anomaly**, i.e. density - 1000 kg m-3: values of 20-28 here mean
1020-1028 kg m-3. Values well below 20 are the Amazon/Orinoco plume (very fresh surface water);
the model's river-mouth cells reach salinities near 0, which give sigma0 down to about -4
(~996 kg m-3, i.e. fresh water) and the correspondingly large positive buoyancies.

In [8]:
print("stores processed per release month:")
print(stats.groupby("month").size().to_string())
print()
print(f"total position samples (obs>0): {stats.n_finite_pos.sum():,.0f}")
print(f"samples falling OUTSIDE the lon/lat bins: {stats.n_out.sum():,.0f}")
print(f"samples rejected as invalid (sal<={sal_min} or temp<={temp_min} or NaN): {stats.n_bad.sum():,.0f}")
print(f"observed lon range {stats.lon_min.min():.2f} .. {stats.lon_max.max():.2f}  "
      f"(bins {lon_bins[0]:.2f} .. {lon_bins[-1]:.2f})")
print(f"observed lat range {stats.lat_min.min():.2f} .. {stats.lat_max.max():.2f}  "
      f"(bins {lat_bins[0]:.2f} .. {lat_bins[-1]:.2f})")
for v in VARS:
    n = stats["n_" + v].sum()
    print(f"{v:8s} n={n:,.0f}  global min={stats['min_'+v].min():.4f}  "
          f"mean={stats['sum_'+v].sum()/n:.4f}  max={stats['max_'+v].max():.4f}")

print()
print("--- sample-level physics check (first 20 stores, all samples) ---")
sub = stores[:20]
sig_all, b_all, sal_all = [], [], []
for f in sub:
    d = xr.open_zarr(f)
    sl = slice(1, None) if drop_first_obs else slice(None)
    dd = d[["lon", "lat", "z", "temp", "sal"]].isel(obs=sl).compute()
    bad = ((~np.isfinite(dd.temp)) | (~np.isfinite(dd.sal))
           | (dd.sal <= sal_min) | (dd.temp <= temp_min))
    dd["temp"] = dd.temp.where(~bad); dd["sal"] = dd.sal.where(~bad)
    p = gsw.p_from_z(-dd.z, dd.lat); SA = gsw.SA_from_SP(dd.sal, p, dd.lon, dd.lat)
    CT = gsw.CT_from_pt(SA, dd.temp)
    sig_all.append(gsw.sigma0(SA, CT).values.ravel())
    b_all.append((-9.81 * (gsw.rho(SA, CT, p) - 1025.0) / 1025.0).values.ravel())
    sal_all.append(dd.sal.values.ravel())
sig_all = np.concatenate(sig_all); b_all = np.concatenate(b_all); sal_all = np.concatenate(sal_all)
q = [0, 1, 5, 25, 50, 75, 95, 99, 100]
print("sigma0 percentiles", dict(zip(q, np.round(np.nanpercentile(sig_all, q), 2))))
print("       -> density  ", dict(zip(q, np.round(np.nanpercentile(sig_all, q) + 1000, 2))), "kg/m3")
print("buoyancy percentiles", dict(zip(q, np.round(np.nanpercentile(b_all, q), 5))), "m/s2")
print("salinity percentiles", dict(zip(q, np.round(np.nanpercentile(sal_all, q), 2))))
print(f"fraction of samples with sal < 5 psu (plume): "
      f"{100*np.nanmean(sal_all < 5):.2f} %")

stores processed per release month:
month
1     132
2     117
3     130
4     126
5     132
6     126
7     129
8     130
9     126
10    130
11    126
12    130

total position samples (obs>0): 11,442,932,431
samples falling OUTSIDE the lon/lat bins: 528
samples rejected as invalid (sal<=0.0 or temp<=0.0 or NaN): 256,849,561
observed lon range -94.96 .. 1.01  (bins -96.94 .. -14.67)
observed lat range -3.63 .. 29.82  (bins -4.38 .. 32.02)
b        n=11,442,513,142  global min=-0.0308  mean=0.0866  max=0.2813
sigma0   n=11,442,513,142  global min=-4.3966  mean=15.9116  max=26.9688

--- sample-level physics check (first 20 stores, all samples) ---


sigma0 percentiles {0: np.float64(-4.17), 1: np.float64(-3.65), 5: np.float64(-3.14), 25: np.float64(3.27), 50: np.float64(21.68), 75: np.float64(22.51), 95: np.float64(23.26), 99: np.float64(23.68), 100: np.float64(26.21)}


       -> density   {0: np.float64(995.83), 1: np.float64(996.35), 5: np.float64(996.86), 25: np.float64(1003.27), 50: np.float64(1021.68), 75: np.float64(1022.51), 95: np.float64(1023.26), 99: np.float64(1023.68), 100: np.float64(1026.21)} kg/m3


buoyancy percentiles {0: np.float64(-0.01894), 1: np.float64(0.01161), 5: np.float64(0.01611), 25: np.float64(0.02331), 50: np.float64(0.03145), 75: np.float64(0.2078), 95: np.float64(0.26914), 99: np.float64(0.27395), 100: np.float64(0.27898)} m/s2


salinity percentiles {0: np.float64(0.0), 1: np.float64(0.18), 5: np.float64(0.65), 25: np.float64(9.3), 50: np.float64(33.96), 75: np.float64(35.13), 95: np.float64(36.03), 99: np.float64(36.44), 100: np.float64(37.67)}
fraction of samples with sal < 5 psu (plume): 19.73 %


## Monthly mean per bin -> zarr

In [9]:
OUTPUTS = {"b":      ("b",      "mean_b_month_v3.zarr"),
           "sigma0": ("sigma0", "mean_sigma0_month_v3.zarr")}
def to_da(v, name):
    mean = SUM[v] / np.where(COUNT[v] > 0, COUNT[v], np.nan)
    return xr.DataArray(mean, dims=("start_month", "lon", "lat"),
                        coords={"start_month": np.arange(1, 13), "lon": lon_c, "lat": lat_c},
                        name=name)

means = {}
for v, (name, fname) in OUTPUTS.items():
    da = to_da(v, name)
    ds = da.to_dataset()
    ds["count"] = xr.DataArray(COUNT[v], dims=("start_month", "lon", "lat"),
                               coords=ds.coords)
    ds.to_zarr(fname, mode="w")
    means[v] = ds
    finite = np.isfinite(da.values)
    print(f"{fname}: {dict(ds.sizes)}  filled bins={finite.sum()}/{finite.size}  "
          f"bin-mean min={np.nanmin(da.values):.4f} median={np.nanmedian(da.values):.4f} "
          f"max={np.nanmax(da.values):.4f}")


mean_b_month_v3.zarr: {'start_month': 12, 'lon': 79, 'lat': 29}  filled bins=11766/27492  bin-mean min=-0.0155 median=0.0170 max=0.2750
mean_sigma0_month_v3.zarr: {'start_month': 12, 'lon': 79, 'lat': 29}  filled bins=11766/27492  bin-mean min=-3.7534 median=23.1573 max=26.1991


## Plots

In [10]:
labs = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

def plot_monthly(da, cmap, label, name, levels=None):
    """3x4 monthly grid, same style as Analysis/Monthly_means_z_temp_sal.ipynb."""
    proj = ccrs.PlateCarree()
    fig, ax = plt.subplots(4, 3, figsize=(12, 10), subplot_kw=dict(projection=proj))
    ax = ax.ravel()
    if levels is None:
        vmin = float(np.nanpercentile(da.values, 1)); vmax = float(np.nanpercentile(da.values, 99))
        levels = np.linspace(vmin, vmax, 13)
    levels = np.asarray(levels, dtype=float)
    ncol = plt.get_cmap(cmap).N if isinstance(cmap, str) else cmap.N
    norm = BoundaryNorm(levels, ncolors=ncol, clip=True)

    for i in range(12):
        dd = da.sel(start_month=i + 1)
        pcm = dd.plot.contourf(x="lon", y="lat", ax=ax[i], transform=proj, cmap=cmap,
                               levels=levels, norm=norm, add_colorbar=False)
        ax[i].coastlines()
        ax[i].add_feature(cf.LAND, facecolor="#8e9497ff", zorder=0)
        ax[i].set_title(labs[i], fontsize=10)
        ax[i].set_xticks(np.arange(np.floor(dd.lon.min()), np.ceil(dd.lon.max()) + 1e-6, 10), crs=proj)
        ax[i].set_yticks(np.arange(np.floor(dd.lat.min()), np.ceil(dd.lat.max()) + 1e-6, 5), crs=proj)
        ax[i].xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f", degree_symbol="\u00b0"))
        ax[i].yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f", degree_symbol="\u00b0"))
        ax[i].tick_params(labelsize=8)
        ax[i].set_xlabel(""); ax[i].set_ylabel("")

    fig.subplots_adjust(left=0.05, right=0.95, bottom=0.08, top=0.90, wspace=0.05, hspace=0.4)
    cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.10)
    cbar.ax.set_xlabel(label, fontsize=10)
    out = os.path.join(fig_dir, f"v2.mean_{name}_LPT_v3.png")
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print("saved", out)
    plt.show()


In [11]:
mean_b   = xr.open_zarr("mean_b_month_v3.zarr")
mean_sig = xr.open_zarr("mean_sigma0_month_v3.zarr")

# levels are uneven on purpose: most water sits at sigma0 21-24, but the Amazon/Orinoco
# plume bins run down to ~-2 (nearly fresh), so the low end needs coarse levels.
plot_monthly(mean_sig.sigma0, cm.cm.dense, r"mean $\sigma_0$ [kg m$^{-3}$]", "sigma0",
             levels=[-4, 0, 4, 8, 12, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26])

saved /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/figures/v2.mean_sigma0_LPT_v3.png


In [12]:
plot_monthly(mean_b.b, cm.cm.balance, r"mean buoyancy [m s$^{-2}$]", "b",
             levels=[-0.01, 0, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05,
                     0.075, 0.10, 0.15, 0.20, 0.28])

saved /work/bk1450/b383184/Amazon/Mercator/notebooks/run_2/analysis/figures/v2.mean_b_LPT_v3.png


In [13]:
cnt = mean_sig["count"].sum(("lon", "lat")).values
for m in range(12):
    print(f"{labs[m]}  stores={int((stats.month==m+1).sum()):4d}  binned samples={cnt[m]:,.0f}")
print(f"TOTAL binned samples: {cnt.sum():,.0f}")

JAN  stores= 132  binned samples=984,585,639
FEB  stores= 117  binned samples=872,790,495
MAR  stores= 130  binned samples=969,799,597
APR  stores= 126  binned samples=939,842,507
MAY  stores= 132  binned samples=984,504,975
JUN  stores= 126  binned samples=939,903,719
JUL  stores= 129  binned samples=962,272,830
AUG  stores= 130  binned samples=969,758,319
SEP  stores= 126  binned samples=939,881,806
OCT  stores= 130  binned samples=969,723,945
NOV  stores= 126  binned samples=939,744,592
DEC  stores= 130  binned samples=969,704,190
TOTAL binned samples: 11,442,512,614
